# FINN Deployment: nnUNet ENet Quantized Model to Xilinx ZCU7EV

<font color="red">**Live FINN tutorial:** We recommend clicking **Cell -> Run All** when you start reading this notebook for "latency hiding".</font>

## Overview

This notebook demonstrates end-to-end deployment of a quantized nnUNet ENet model via the FINN compiler to a Xilinx ZCU7EV FPGA device (**xczu7ev-ffvc1156-2-e**). 

The model being deployed is: `nnUNetTrainerENetQuant_3a_quant_O4_int8_no_asym.onnx`

Key features of this flow:
- Automatic folding and optimization using FINN's built-in strategies
- Comprehensive diagnostics including layer analysis and resource estimates
- Performance and resource utilization reporting at each stage
- Support for streaming dataflow accelerator generation
- Hardware deployment verification

## Outline
1. [Import Required Libraries and Setup](#setup)
2. [Load and Validate ONNX Model](#load_model)
3. [Configure FINN Build Environment](#configure_build)
4. [Model Preparation and Graph Transformation](#prepare_model)
5. [Generate Estimation Reports](#estimate_reports)
6. [Launch Hardware Build (IP, Synthesis, Simulation)](#hardware_build)
7. [Analyze Resource and Performance Estimates](#analyze_estimates)
8. [(Optional) Generate Bitstream](#generate_bitstream)

In [ ]:
# FINN environment setup: add source trees to Python path.
import sys as _sys
for _p in [
    "/home/thelegendiv/finn/src",
    "/home/thelegendiv/finn/deps/qonnx/src",
    "/home/thelegendiv/finn/deps/brevitas/src",
    "/home/thelegendiv/finn/deps/pyverilator",
    "/home/thelegendiv/finn/deps/finn-experimental",
]:
    if _p not in _sys.path:
        _sys.path.insert(0, _p)
del _sys, _p
import onnx
import torch


## 1. Import Required Libraries and Setup <a id="setup"></a>

In [2]:
import os
import sys
import json
import logging
import shutil
import subprocess
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any
import warnings

# FINN imports
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
from finn.util.visualization import showInNetron

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=DeprecationWarning)

print("=" * 80)
print("FINN END-TO-END DEPLOYMENT PIPELINE FOR NNUNET ENET")
print("Target Device: Xilinx ZCU7EV (xczu7ev-ffvc1156-2-e)")
print("=" * 80)
print(f"Notebook started at: {datetime.now().isoformat()}")

FINN END-TO-END DEPLOYMENT PIPELINE FOR NNUNET ENET
Target Device: Xilinx ZCU7EV (xczu7ev-ffvc1156-2-e)
Notebook started at: 2026-07-27T17:34:12.351134


In [ ]:
# Define paths
PROJECT_ROOT = os.getcwd()

# MODEL selection - change this to switch between model variants:
#   quantEnet_debug_full.onnx     : debug net, channels=4, no residuals
#   quantEnet_finn_v1.onnx        : production net, channels=(20,72,144,72,20), residuals
#   quantEnet_finn_v1_no_res.onnx : production net without residuals
MODEL_FILENAME = "quantEnet_finn_v1.onnx"
MODEL_PATH = os.path.join(PROJECT_ROOT, MODEL_FILENAME)

# Create output directories for build artifacts
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_BASE = os.path.join(PROJECT_ROOT, "finn_deployment_outputs")
OUTPUT_ESTIMATES_DIR = os.path.join(OUTPUT_BASE, f"estimates_{TIMESTAMP}")
OUTPUT_HWBUILD_DIR   = os.path.join(OUTPUT_BASE, f"hwbuild_{TIMESTAMP}")
OUTPUT_BITSTREAM_DIR = os.path.join(OUTPUT_BASE, f"bitstream_{TIMESTAMP}")

for output_dir in [OUTPUT_BASE, OUTPUT_ESTIMATES_DIR, OUTPUT_HWBUILD_DIR, OUTPUT_BITSTREAM_DIR]:
    os.makedirs(output_dir, exist_ok=True)

# FPGA Configuration
TARGET_DEVICE  = "xczu7ev-ffvc1156-2-e"
TARGET_BOARD   = "ZCU7EV"
XILINX_VERSION = "2022.2"

logger.info(f"Model path: {MODEL_PATH}")
logger.info(f"Output base directory: {OUTPUT_BASE}")
logger.info(f"Target device: {TARGET_DEVICE}")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model file not found: {MODEL_PATH}")
else:
    model_size_mb = os.path.getsize(MODEL_PATH) / (1024**2)
    logger.info(f"Model file found: {model_size_mb:.2f} MB")


2026-07-27 17:34:12,369 - __main__ - INFO - Model path: /home/thelegendiv/finn/notebooks/enet/quantEnet_debug_full.onnx
2026-07-27 17:34:12,371 - __main__ - INFO - Output base directory: /home/thelegendiv/finn/notebooks/enet/finn_deployment_outputs
2026-07-27 17:34:12,372 - __main__ - INFO - Target device: xczu7ev-ffvc1156-2-e
2026-07-27 17:34:12,373 - __main__ - INFO - Model file found: 0.05 MB


## 2. Load and Validate ONNX Model <a id="load_model"></a>

In this section we'll import the ONNX model into FINN using the `ModelWrapper` class and extract key information about its structure, inputs, outputs, and layer composition.

In [4]:
# Load ONNX model with FINN ModelWrapper
model = ModelWrapper(MODEL_PATH)

logger.info("=" * 60)
logger.info("ONNX MODEL STRUCTURE ANALYSIS")
logger.info("=" * 60)

# Extract input/output tensor information
input_tensor_name = model.graph.input[0].name
output_tensor_name = model.graph.output[0].name

input_shape = model.get_tensor_shape(input_tensor_name)
output_shape = model.get_tensor_shape(output_tensor_name)

input_dtype = model.get_tensor_datatype(input_tensor_name)
output_dtype = model.get_tensor_datatype(output_tensor_name)

logger.info(f"Number of inputs: {len(model.graph.input)}")
logger.info(f"Number of outputs: {len(model.graph.output)}")
logger.info(f"Number of nodes: {len(model.graph.node)}")

logger.info(f"\nInput Tensor:")
logger.info(f"  Name: {input_tensor_name}")
logger.info(f"  Shape: {input_shape}")
logger.info(f"  Data Type: {input_dtype}")

logger.info(f"\nOutput Tensor:")
logger.info(f"  Name: {output_tensor_name}")
logger.info(f"  Shape: {output_shape}")
logger.info(f"  Data Type: {output_dtype}")

# Extract layer information
operation_types = [x.op_type for x in model.graph.node]
unique_ops = list(set(operation_types))
op_counts = {}
for op in unique_ops:
    op_counts[op] = operation_types.count(op)

logger.info(f"\nLayer Operation Types ({len(unique_ops)} unique types):")
for op_type, count in sorted(op_counts.items(), key=lambda x: x[1], reverse=True):
    logger.info(f"  {op_type}: {count}")

# Save model information to JSON
model_info = {
    "model_path": MODEL_PATH,
    "num_inputs": len(model.graph.input),
    "num_outputs": len(model.graph.output),
    "num_nodes": len(model.graph.node),
    "input_tensor": {
        "name": input_tensor_name,
        "shape": input_shape,
        "dtype": str(input_dtype)
    },
    "output_tensor": {
        "name": output_tensor_name,
        "shape": output_shape,
        "dtype": str(output_dtype)
    },
    "operation_types": op_counts,
    "timestamp": datetime.now().isoformat()
}

model_info_path = os.path.join(OUTPUT_BASE, f"model_info_{TIMESTAMP}.json")
with open(model_info_path, 'w') as f:
    json.dump(model_info, f, indent=2)
logger.info(f"\nModel information saved to: {model_info_path}")

2026-07-27 17:34:12,470 - __main__ - INFO - ============================================================
2026-07-27 17:34:12,470 - __main__ - INFO - ONNX MODEL STRUCTURE ANALYSIS
2026-07-27 17:34:12,471 - __main__ - INFO - ============================================================
2026-07-27 17:34:12,473 - __main__ - INFO - Number of inputs: 1
2026-07-27 17:34:12,474 - __main__ - INFO - Number of outputs: 1
2026-07-27 17:34:12,475 - __main__ - INFO - Number of nodes: 132
2026-07-27 17:34:12,475 - __main__ - INFO - 
Input Tensor:
2026-07-27 17:34:12,476 - __main__ - INFO -   Name: global_in
2026-07-27 17:34:12,476 - __main__ - INFO -   Shape: [1, 1, 64, 64]
2026-07-27 17:34:12,477 - __main__ - INFO -   Data Type: INT8
2026-07-27 17:34:12,478 - __main__ - INFO - 
Output Tensor:
2026-07-27 17:34:12,478 - __main__ - INFO -   Name: global_out
2026-07-27 17:34:12,479 - __main__ - INFO -   Shape: [1, 2, 64, 64]
2026-07-27 17:34:12,479 - __main__ - INFO -   Data Type: INT8
2026-07-27 17:34:1

## 3. Configure FINN Build Environment <a id="configure_build"></a>

In this section we configure the FINN build parameters including the target FPGA device, performance targets, and optimization strategies.

In [5]:
# FPGA device specifications
fpga_part = "xczu7ev-ffvc1156-2-e"

# Build configuration parameters
# These are aggressive settings for initial testing - adjust based on results
build_params = {
    "fpga_part": fpga_part,
    "board": TARGET_BOARD,
    "synth_clk_period_ns": 10.0,  # Target 100 MHz clock
    "target_fps": 10000,           # Target frames per second (test setting)
    "mvau_wwidth_max": 80,         # Maximum width for MVAU dataflow operations
}

logger.info("=" * 60)
logger.info("FINN BUILD CONFIGURATION")
logger.info("=" * 60)
logger.info(f"FPGA Part: {build_params['fpga_part']}")
logger.info(f"Target Board: {build_params['board']}")
logger.info(f"Target Clock Period (ns): {build_params['synth_clk_period_ns']}")
logger.info(f"Target Performance (fps): {build_params['target_fps']}")
logger.info(f"Max MVAU Width: {build_params['mvau_wwidth_max']}")

# Save build config to JSON for reference
build_config_path = os.path.join(OUTPUT_BASE, f"build_config_{TIMESTAMP}.json")
with open(build_config_path, 'w') as f:
    json.dump(build_params, f, indent=2)
logger.info(f"\nBuild configuration saved to: {build_config_path}")

2026-07-27 17:34:12,494 - __main__ - INFO - ============================================================
2026-07-27 17:34:12,504 - __main__ - INFO - FINN BUILD CONFIGURATION
2026-07-27 17:34:12,506 - __main__ - INFO - ============================================================
2026-07-27 17:34:12,508 - __main__ - INFO - FPGA Part: xczu7ev-ffvc1156-2-e
2026-07-27 17:34:12,509 - __main__ - INFO - Target Board: ZCU7EV
2026-07-27 17:34:12,510 - __main__ - INFO - Target Clock Period (ns): 10.0
2026-07-27 17:34:12,511 - __main__ - INFO - Target Performance (fps): 10000
2026-07-27 17:34:12,512 - __main__ - INFO - Max MVAU Width: 80
2026-07-27 17:34:12,516 - __main__ - INFO - 
Build configuration saved to: /home/thelegendiv/finn/notebooks/enet/finn_deployment_outputs/build_config_20260727_173412.json


## 4. Model Preparation and Graph Transformation <a id="prepare_model"></a>

Before hardware synthesis, we need to prepare the ONNX model by applying several graph transformations to ensure all tensors have statically-defined shapes and proper datatype annotations.

In [6]:
logger.info("=" * 60)
logger.info("APPLYING MODEL TRANSFORMATIONS")
logger.info("=" * 60)

# Save original model
original_model_path = os.path.join(OUTPUT_BASE, f"model_original_{TIMESTAMP}.onnx")
model.save(original_model_path)
logger.info(f"Original model saved: {original_model_path}")

# Apply transformations
transforms = [
    ("InferShapes", InferShapes()),
    ("FoldConstants", FoldConstants()),
    ("GiveUniqueNodeNames", GiveUniqueNodeNames()),
    ("GiveReadableTensorNames", GiveReadableTensorNames()),
    ("InferDataTypes", InferDataTypes()),
    ("RemoveStaticGraphInputs", RemoveStaticGraphInputs()),
]

for transform_name, transform in transforms:
    logger.info(f"Applying: {transform_name}...")
    model = model.transform(transform)
    logger.info(f"  âœ“ {transform_name} completed")

# Save transformed model
transformed_model_path = os.path.join(OUTPUT_BASE, f"model_transformed_{TIMESTAMP}.onnx")
model.save(transformed_model_path)
logger.info(f"\nTransformed model saved: {transformed_model_path}")

# Extract updated model information
input_node_count_after = len(model.graph.input)
output_node_count_after = len(model.graph.output)
node_count_after = len(model.graph.node)

logger.info(f"\nModel Statistics After Transformation:")
logger.info(f"  Inputs: {input_node_count_after}")
logger.info(f"  Outputs: {output_node_count_after}")
logger.info(f"  Nodes: {node_count_after}")

2026-07-27 17:34:12,524 - __main__ - INFO - ============================================================
2026-07-27 17:34:12,524 - __main__ - INFO - APPLYING MODEL TRANSFORMATIONS
2026-07-27 17:34:12,533 - __main__ - INFO - ============================================================
2026-07-27 17:34:12,541 - __main__ - INFO - Original model saved: /home/thelegendiv/finn/notebooks/enet/finn_deployment_outputs/model_original_20260727_173412.onnx
2026-07-27 17:34:12,542 - __main__ - INFO - Applying: InferShapes...
2026-07-27 17:34:12,645 - __main__ - INFO -   âœ“ InferShapes completed
2026-07-27 17:34:12,646 - __main__ - INFO - Applying: FoldConstants...
2026-07-27 17:34:12,727 - __main__ - INFO -   âœ“ FoldConstants completed
2026-07-27 17:34:12,728 - __main__ - INFO - Applying: GiveUniqueNodeNames...
2026-07-27 17:34:12,759 - __main__ - INFO -   âœ“ GiveUniqueNodeNames completed
2026-07-27 17:34:12,760 - __main__ - INFO - Applying: GiveReadableTensorNames...
2026-07-27 17:34:13,311 - _

## 5. Generate Estimation Reports <a id="estimate_reports"></a>

First, we'll launch a build that only generates analytical estimation reports without any synthesis. This is fast and provides insights into expected performance and resource utilization.

In [ ]:
logger.info("=" * 60)
logger.info("LAUNCHING FULL ESTIMATION BUILD (all estimate_only_dataflow_steps)")
logger.info("=" * 60)

# Run the full estimation pipeline: no stop_step, so FINN goes all the way
# through step_generate_estimate_reports and produces resource/performance JSON.
cfg_estimates = build.DataflowBuildConfig(
    output_dir=OUTPUT_ESTIMATES_DIR,
    mvau_wwidth_max=build_params["mvau_wwidth_max"],
    target_fps=build_params["target_fps"],
    synth_clk_period_ns=build_params["synth_clk_period_ns"],
    fpga_part=build_params["fpga_part"],
    steps=build_cfg.estimate_only_dataflow_steps,
    generate_outputs=[build_cfg.DataflowOutputType.ESTIMATE_REPORTS],
    save_intermediate_models=True,
    enable_build_pdb_debug=False,
    verbose=True,
)

logger.info(f"Build output directory: {OUTPUT_ESTIMATES_DIR}")
logger.info("Starting FINN estimation build...")

try:
    build.build_dataflow_cfg(transformed_model_path, cfg_estimates)
    logger.info("Estimation build completed successfully")
except Exception as e:
    logger.error(f"Estimation build failed: {str(e)}")
    import traceback; traceback.print_exc()
    raise


2026-07-27 17:34:13,536 - __main__ - INFO - ============================================================
2026-07-27 17:34:13,538 - __main__ - INFO - LAUNCHING DEBUG BUILD: STOP AFTER CONVERT_TO_HW
2026-07-27 17:34:13,539 - __main__ - INFO - ============================================================
2026-07-27 17:34:13,540 - __main__ - INFO - Build output directory: /home/thelegendiv/finn/notebooks/enet/finn_deployment_outputs/estimates_20260727_173412
2026-07-27 17:34:13,540 - __main__ - INFO - Starting FINN debug build until step_convert_to_hw...


Building dataflow accelerator from /home/thelegendiv/finn/notebooks/enet/finn_deployment_outputs/model_transformed_20260727_173412.onnx
Intermediate outputs will be generated in /tmp/finn_dev_thelegendiv
Final outputs will be generated in /home/thelegendiv/finn/notebooks/enet/finn_deployment_outputs/estimates_20260727_173412
Build log is at /home/thelegendiv/finn/notebooks/enet/finn_deployment_outputs/estimates_20260727_173412/build_dataflow.log
Running step: step_qonnx_to_finn [1/4]
Running step: step_tidy_up [2/4]
Running step: step_streamline [3/4]


/home/thelegendiv/finn/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:127: UserWarning: Assuming 4D input is NCHW
  warnings.warn("Assuming 4D input is NCHW")


Running step: step_convert_to_hw [4/4]


2026-07-27 17:34:33,870 - __main__ - INFO - âœ“ Debug build reached step_convert_to_hw successfully


Completed successfully


In [ ]:
from finn.util.visualization import showInNetron

# Currently-deployed model: fully specialized HW dataflow graph (325 nodes) from the
# quantEnet_O8_native stitched-IP build (container died mid-Vivado-stitching, but this
# intermediate export from step_hw_ipgen survived on the bind-mounted enet/ directory).
# This is the model right after per-node HLS/RTL synthesis (step_hw_ipgen), the last
# step to fully complete before the build was interrupted.
onnx_path = (
    "finn_deployment_outputs/stitched_ip_quantEnet_O8_native_20260729_123711/"
    "intermediate_models/step_hw_ipgen.onnx"
)

showInNetron(onnx_path)


2026-07-27 17:34:33,879 - netron.server - INFO - Serving 'finn_deployment_outputs/estimates_20260727_171804/intermediate_models/step_streamline.onnx' at http://0.0.0.0:8081


In [ ]:
# Load the step_convert_to_hw intermediate model to check what was not converted
_convert_hw_path = os.path.join(OUTPUT_ESTIMATES_DIR, "intermediate_models", "step_convert_to_hw.onnx")
if os.path.exists(_convert_hw_path):
    _hw_model = ModelWrapper(_convert_hw_path)
else:
    logger.warning("step_convert_to_hw.onnx not found, falling back to transformed model")
    _hw_model = model  # fallback

FINN_CUSTOM_DOMAIN = "finn.custom_op.fpgadataflow"
HW_OP_NAMES = {
    "MVAU", "Thresholding", "FMPadding", "ConvolutionInputGenerator",
    "StreamingMaxPool", "StreamingDataWidthConverter", "StreamingFIFO",
    "ConvTranspose", "Pool_Batch", "AddStreams", "AddStreams_Batch",
    "FMPadding_Batch", "ChannelwiseOp_Batch",
}

hw_counts, non_hw_counts = {}, {}
for _n in _hw_model.graph.node:
    _is_hw = (_n.op_type in HW_OP_NAMES) or (_n.domain == FINN_CUSTOM_DOMAIN)
    _d = hw_counts if _is_hw else non_hw_counts
    _d[_n.op_type] = _d.get(_n.op_type, 0) + 1

print("=" * 80)
print("AFTER step_convert_to_hw")
print("=" * 80)
print(f"Total nodes: {len(_hw_model.graph.node)}")
n_hw = sum(hw_counts.values())
n_non = sum(non_hw_counts.values())
print(f"HW ops converted ({n_hw} nodes):")
for k, v in sorted(hw_counts.items()):
    print(f"  {k}: {v}")
print(f"Non-HW blockers ({n_non} nodes):")
for k, v in sorted(non_hw_counts.items()):
    print(f"  {k}: {v}")
if not non_hw_counts:
    print("  (none -- all nodes converted to HW ops! Estimation should be complete.")


NON-HW OPS REMAINING AFTER step_convert_to_hw
Quant Quant_0
Quant Quant_1
Quant Quant_2
Quant Quant_3
Quant Quant_4
Quant Quant_5
Quant Quant_6
Quant Quant_7
Quant Quant_8
Quant Quant_9
Quant Quant_10
Quant Quant_11
Quant Quant_12
Quant Quant_13
Quant Quant_14
Quant Quant_15
Quant Quant_16
Quant Quant_17
Quant Quant_18
Quant Quant_19
Quant Quant_20
Quant Quant_21
Quant Quant_22
Quant Quant_23
Quant Quant_24
Quant Quant_25
Quant Quant_26
Quant Quant_27
Conv Conv_0
BatchNormalization BatchNormalization_0
Relu Relu_0
Quant Quant_28
MaxPool MaxPool_0
Conv Conv_1
BatchNormalization BatchNormalization_1
Relu Relu_1
Quant Quant_29
Conv Conv_2
BatchNormalization BatchNormalization_2
Relu Relu_2
Quant Quant_30
Conv Conv_3
BatchNormalization BatchNormalization_3
Relu Relu_3
Quant Quant_31
Conv Conv_4
BatchNormalization BatchNormalization_4
Relu Relu_4
Quant Quant_32
Conv Conv_5
BatchNormalization BatchNormalization_5
Relu Relu_5
Quant Quant_33
Conv Conv_6
BatchNormalization BatchNormalization_6


## 6. Analyze Resource and Performance Estimates <a id="analyze_estimates"></a>

Now let's examine the detailed estimation reports to understand the expected resource utilization and performance characteristics.

In [10]:
def read_json_report(filepath):
    """Helper function to read JSON report files"""
    with open(filepath, 'r') as f:
        return json.load(f)

# Read network performance estimates
perf_report_path = os.path.join(report_dir, "estimate_network_performance.json")
if os.path.exists(perf_report_path):
    perf_report = read_json_report(perf_report_path)
    logger.info("=" * 60)
    logger.info("NETWORK PERFORMANCE ESTIMATES")
    logger.info("=" * 60)
    for key, value in perf_report.items():
        logger.info(f"{key}: {value}")

# Read layer cycle estimates
layer_cycles_path = os.path.join(report_dir, "estimate_layer_cycles.json")
if os.path.exists(layer_cycles_path):
    layer_cycles = read_json_report(layer_cycles_path)
    logger.info("\n" + "=" * 60)
    logger.info("LAYER-BY-LAYER CYCLE ESTIMATES")
    logger.info("=" * 60)
    total_cycles = 0
    for layer_name, cycles in layer_cycles.items():
        logger.info(f"{layer_name}: {cycles} cycles")
        total_cycles += cycles
    logger.info(f"Total (sum): {total_cycles} cycles")

# Read resource estimates
resources_path = os.path.join(report_dir, "estimate_layer_resources.json")
if os.path.exists(resources_path):
    resources = read_json_report(resources_path)
    logger.info("\n" + "=" * 60)
    logger.info("LAYER-BY-LAYER RESOURCE ESTIMATES")
    logger.info("=" * 60)
    
    # Compute totals
    total_lut = 0
    total_bram = 0
    total_dsp = 0
    
    for layer_name, layer_resources in resources.items():
        logger.info(f"\n{layer_name}:")
        for resource_type, value in layer_resources.items():
            logger.info(f"  {resource_type}: {value}")
            if resource_type == "LUT":
                total_lut += value
            elif resource_type == "BRAM":
                total_bram += value
            elif resource_type == "DSP":
                total_dsp += value
    
    logger.info(f"\n{'TOTALS':^60}")
    logger.info(f"Total LUT: {total_lut}")
    logger.info(f"Total BRAM: {total_bram}")
    logger.info(f"Total DSP: {total_dsp}")
    
    # ZCU7EV FPGA resource capacity
    xczu7ev_resources = {
        "LUT": 48000,
        "BRAM": 216,
        "DSP": 192,
    }
    
    logger.info(f"\nXCZU7EV Available Resources:")
    logger.info(f"LUT: {xczu7ev_resources['LUT']} (usage: {100*total_lut/xczu7ev_resources['LUT']:.1f}%)")
    logger.info(f"BRAM: {xczu7ev_resources['BRAM']} (usage: {100*total_bram/xczu7ev_resources['BRAM']:.1f}%)")
    logger.info(f"DSP: {xczu7ev_resources['DSP']} (usage: {100*total_dsp/xczu7ev_resources['DSP']:.1f}%)")

NameError: name 'report_dir' is not defined

## 7. Launch Hardware Build (Stitched IP, Synthesis, RTL Simulation) <a id="hardware_build"></a>

<font color="red">**Note:** This section launches a full hardware build with HLS synthesis and RTL simulation. This is significantly slower (10-30 minutes) than the estimation-only pass above. It generates:
- Stitched IP design in Vivado format
- Out-of-context synthesis with post-synthesis resource and timing estimates
- RTL simulation performance measurements

Skip this section if you only want the quick estimates. Uncomment the cell below to run it.</font>

In [ ]:
# HARDWARE BUILD CONFIGURATION
# Uncomment the section below to run the full hardware build

# logger.info("=" * 60)
# logger.info("LAUNCHING FULL HARDWARE BUILD")
# logger.info("=" * 60)

# cfg_hwbuild = build.DataflowBuildConfig(
#     output_dir=OUTPUT_HWBUILD_DIR,
#     mvau_wwidth_max=build_params["mvau_wwidth_max"],
#     target_fps=build_params["target_fps"],
#     synth_clk_period_ns=build_params["synth_clk_period_ns"],
#     fpga_part=build_params["fpga_part"],
#     generate_outputs=[
#         build_cfg.DataflowOutputType.STITCHED_IP,
#         build_cfg.DataflowOutputType.RTLSIM_PERFORMANCE,
#         build_cfg.DataflowOutputType.OOC_SYNTH,
#     ]
# )

# logger.info(f"Build output directory: {OUTPUT_HWBUILD_DIR}")
# logger.info("Starting FINN hardware build (this will take 10-30 minutes)...")
# logger.info("WARNING: This includes HLS synthesis and RTL simulation\n")

# try:
#     build.build_dataflow_cfg(transformed_model_path, cfg_hwbuild)
#     logger.info("âœ“ Hardware build completed successfully")
# except Exception as e:
#     logger.error(f"Hardware build failed: {str(e)}")
#     raise

logger.info("HARDWARE BUILD: Skipped (comment out section to enable)")
logger.info("To enable hardware build, uncomment the cell above and re-run.")

## 8. (Optional) Generate Bitstream for Deployment <a id="generate_bitstream"></a>

<font color="red">**Note:** Bitstream generation requires full Vivado synthesis and place-and-route, which takes 20-60 minutes. This section is for generating a deployable bitstream for the ZCU7EV board. Skip if you only need estimates.</font>

In [ ]:
# BITSTREAM GENERATION CONFIGURATION
# Uncomment the section below to run the full bitstream generation

# logger.info("=" * 60)
# logger.info("LAUNCHING BITSTREAM GENERATION")
# logger.info("=" * 60)

# cfg_bitstream = build.DataflowBuildConfig(
#     output_dir=OUTPUT_BITSTREAM_DIR,
#     mvau_wwidth_max=build_params["mvau_wwidth_max"],
#     target_fps=build_params["target_fps"],
#     synth_clk_period_ns=build_params["synth_clk_period_ns"],
#     board=build_params["board"],
#     shell_flow_type=build_cfg.ShellFlowType.VIVADO_ZYNQ,
#     generate_outputs=[
#         build_cfg.DataflowOutputType.BITFILE,
#         build_cfg.DataflowOutputType.PYNQ_DRIVER,
#         build_cfg.DataflowOutputType.DEPLOYMENT_PACKAGE,
#     ]
# )

# logger.info(f"Build output directory: {OUTPUT_BITSTREAM_DIR}")
# logger.info("Starting FINN bitstream generation (this will take 20-60 minutes)...")
# logger.info("WARNING: This includes full Vivado synthesis and place-and-route\n")

# try:
#     build.build_dataflow_cfg(transformed_model_path, cfg_bitstream)
#     logger.info("âœ“ Bitstream generation completed successfully")
#     
#     # Check that bitstream files exist
#     bitfile_dir = os.path.join(OUTPUT_BITSTREAM_DIR, "bitfile")
#     deploy_dir = os.path.join(OUTPUT_BITSTREAM_DIR, "deploy")
#     
#     if os.path.exists(bitfile_dir):
#         logger.info(f"\nBitfile outputs in {bitfile_dir}:")
#         for f in os.listdir(bitfile_dir):
#             logger.info(f"  - {f}")
#     
#     if os.path.exists(deploy_dir):
#         logger.info(f"\nDeployment package in {deploy_dir}:")
#         for f in os.listdir(deploy_dir):
#             logger.info(f"  - {f}")
# 
# except Exception as e:
#     logger.error(f"Bitstream generation failed: {str(e)}")
#     raise

logger.info("BITSTREAM GENERATION: Skipped (comment out section to enable)")
logger.info("To enable bitstream generation, uncomment the cell above and re-run.")

## Deployment Summary and Next Steps

In [ ]:
logger.info("=" * 80)
logger.info("FINN DEPLOYMENT PIPELINE - EXECUTION SUMMARY")
logger.info("=" * 80)

logger.info(f"\nProject Information:")
logger.info(f"  Target Device: {TARGET_DEVICE}")
logger.info(f"  Target Board: {TARGET_BOARD}")
logger.info(f"  Model: {MODEL_PATH}")
logger.info(f"  Timestamp: {TIMESTAMP}")

logger.info(f"\nGenerated Artifacts:")
logger.info(f"  Base Output Directory: {OUTPUT_BASE}")
logger.info(f"  Estimation Reports: {OUTPUT_ESTIMATES_DIR}")
logger.info(f"  Hardware Build: {OUTPUT_HWBUILD_DIR}")
logger.info(f"  Bitstream Build: {OUTPUT_BITSTREAM_DIR}")

logger.info(f"\nKey Files Generated:")
logger.info(f"  Model Info: {model_info_path}")
logger.info(f"  Build Config: {build_config_path}")
logger.info(f"  Original Model: {original_model_path}")
logger.info(f"  Transformed Model: {transformed_model_path}")
logger.info(f"  Report Directory: {report_dir}")

logger.info(f"\nNext Steps:")
logger.info(f"  1. Review estimation reports in: {report_dir}")
logger.info(f"  2. If estimates are acceptable, enable Section 7 (Hardware Build)")
logger.info(f"  3. If bitstream is needed, enable Section 8 (Bitstream Generation)")
logger.info(f"  4. Copy generated bitstream to ZCU7EV board for deployment")

logger.info(f"\n" + "=" * 80)
logger.info(f"Pipeline completed at: {datetime.now().isoformat()}")
logger.info("=" * 80)